In [0]:
# ECG waveform bronze: CAMM DICOM landing -> map_ecg_study / map_ecg_waveform / map_ecg_annotation
#
# ADF owns the fetch (EcgLoad -> EcgFetchBatch on the ADCVnet SHIR; Databricks has no
# network route to CAMM). ADF claims manifest rows and lands one DICOM per ECG in
#   /Volumes/1_inland/sectra/vone/<landing_root>/batches/<batch_id>/<study_uid>__<device_id>__i<n>of<count>.dcm
# where batch_id is the claiming EcgFetchBatch RunId, so a file proves which claim wrote it.
# This notebook owns everything after the file lands:
#   action=setup           manifest table + candidates view (idempotent)
#   action=ingest          reconcile claimed -> downloaded, parse, publish per ECG generation, mark parsed
#   action=reparse_failed  parse_failed -> downloaded (run after a parser fix, then ingest)
#
# Pure parsing lives in ecg_dicom.py (unit-tested locally). Promotion is human-gated:
# ~/docs/superpowers/plans/2026-09-29-ecg-waveform-bronze.md, Task 10.

In [0]:
%pip install pydicom==3.0.1

In [0]:
import json
import os
import re
from datetime import datetime, timezone

import pandas as pd
from pyspark import cloudpickle
from pyspark.sql import functions as F

import ecg_dicom

# Ship ecg_dicom to executors by value: they cannot import workspace files by name.
cloudpickle.register_pickle_by_value(ecg_dicom)


def _widget_text(name, default):
    try:
        dbutils.widgets.text(name, default)
    except Exception:
        pass
    try:
        v = dbutils.widgets.get(name)
    except Exception:
        v = default
    return (v or default).strip()


TARGET_SCHEMA = _widget_text("target_schema", "8_dev.bronze")
ALLOW_PROD_WRITE = _widget_text("allow_production_write", "false").lower() == "true"
assert TARGET_SCHEMA.startswith("8_dev.") or ALLOW_PROD_WRITE, (
    f"Refusing to write {TARGET_SCHEMA} without allow_production_write=true")


def _control_schema(target):
    return "6_mgmt.bronze" if target == "4_prod.bronze" else target


CONTROL_SCHEMA = _control_schema(TARGET_SCHEMA)
ACTION = _widget_text("action", "ingest")
LANDING_ROOT = _widget_text("landing_root", "ecg_camm_dev")
MAX_FILES = int(_widget_text("max_files", "20000"))
CLAIM_TIMEOUT_HOURS = int(_widget_text("claim_timeout_hours", "6"))
MAX_ATTEMPTS = int(_widget_text("max_attempts", "5"))
ADF_RUN_ID = _widget_text("adf_run_id", "manual")
assert ACTION in ("setup", "ingest", "reparse_failed"), ACTION
assert (LANDING_ROOT == "ecg_camm") == (TARGET_SCHEMA == "4_prod.bronze"), (
    "the prod target pairs only with the prod landing folder, and vice versa")

LANDING_BASE = f"/Volumes/1_inland/sectra/vone/{LANDING_ROOT}"
MANIFEST = f"{CONTROL_SCHEMA}.ecg_camm_manifest"
CANDIDATES = f"{CONTROL_SCHEMA}.ecg_camm_candidates_v"
STAGE = f"{CONTROL_SCHEMA}.ecg_parse_stage"      # parsed records of this run
CLAIMS = f"{CONTROL_SCHEMA}.ecg_parse_claims"    # manifest snapshot of the rows this run parses
PUBLISH = f"{CONTROL_SCHEMA}.ecg_parse_publish"  # (ECG, FILE_SHA256) generations this run publishes
T_STUDY = f"{TARGET_SCHEMA}.map_ecg_study"
T_WAVEFORM = f"{TARGET_SCHEMA}.map_ecg_waveform"
T_ANNOTATION = f"{TARGET_SCHEMA}.map_ecg_annotation"
SENTINEL_FLOOR = "1901-01-01"
MRN_ALIAS_TYPE_CD = 10  # code_set 4, CDF_MEANING 'MRN'
FLUSH_RECORDS = 500     # mapInPandas output rows per yielded frame (waveform rows carry ~5k samples)

In [0]:
# Setup: manifest (claim queue shared by ADF daily + backfill runs) and the candidates view.
#
# Two independent versions per ECG decide what work is outstanding:
#   imaging:    source_updt_dt_tm (latest seen)  vs claim_source_updt_dt_tm (what the last claim fetched)
#               -> newer imaging re-queues a CAMM fetch (status pending), even if it arrived mid-claim.
#   enrichment: enrichment_hash (latest CV_PROC facts) vs applied_enrichment_hash (what bronze carries)
#               -> newer enrichment re-parses the already-landed file (status downloaded), no CAMM call.
ENRICHMENT_TYPE = ("STRUCT<person_id: BIGINT, cv_proc_id: BIGINT, im_study_id: BIGINT, "
                   "im_acquired_study_id: BIGINT, encntr_id: BIGINT, order_id: BIGINT, "
                   "mill_accession: STRING, proc_status_cd: BIGINT, trust: STRING, action_dt_tm: TIMESTAMP>")

MANIFEST_DDL = f"""
CREATE TABLE IF NOT EXISTS {MANIFEST} (
  study_instance_uid STRING NOT NULL,
  device_id BIGINT NOT NULL,
  file_stem STRING NOT NULL,
  action_dt_tm TIMESTAMP,
  source_updt_dt_tm TIMESTAMP,
  enrichment {ENRICHMENT_TYPE},
  enrichment_hash STRING,
  status STRING NOT NULL,
  attempts INT NOT NULL,
  refetch_count INT NOT NULL,
  batch_id STRING,
  claimed_at TIMESTAMP,
  claim_source_updt_dt_tm TIMESTAMP,
  downloaded_at TIMESTAMP,
  file_path STRING,
  instance_number INT,
  instance_count INT,
  parsed_at TIMESTAMP,
  parser_version STRING,
  applied_enrichment_hash STRING,
  last_error STRING,
  first_seen_at TIMESTAMP NOT NULL,
  updated_at TIMESTAMP NOT NULL)
CLUSTER BY (status, action_dt_tm)
COMMENT 'ECG CAMM fetch queue. status: pending -> claimed (ADF EcgFetchBatch, batch_id = its RunId) -> downloaded (file landed in batches/<batch_id>) -> parsed | parse_failed; failed_permanent after max attempts. Seeded by EcgLoad from ecg_camm_candidates_v.'
TBLPROPERTIES ('delta.enableRowTracking' = 'true', 'delta.enableDeletionVectors' = 'true')
"""

CANDIDATES_DDL = f"""
CREATE OR REPLACE VIEW {CANDIDATES}
COMMENT 'One row per fetchable CAMM ECG (study UID x device) from Millennium CV_PROC -> IM_STUDY -> IM_ACQUIRED_STUDY, with the CV_PROC enrichment bronze carries.'
AS
WITH j AS (
  SELECT trim(s.STUDY_UID) AS study_instance_uid,
         CAST(a.IM_DEVICE_ID AS BIGINT) AS device_id,
         CAST(p.ACTION_DT_TM AS TIMESTAMP) AS action_dt_tm,
         greatest(CAST(s.UPDT_DT_TM AS TIMESTAMP), CAST(a.UPDT_DT_TM AS TIMESTAMP)) AS source_updt_dt_tm,
         CAST(a.IM_ACQUIRED_STUDY_ID AS BIGINT) AS im_acquired_study_id,
         CAST(p.CV_PROC_ID AS BIGINT) AS cv_proc_id,
         CAST(named_struct(
           'person_id', p.PERSON_ID, 'cv_proc_id', p.CV_PROC_ID, 'im_study_id', s.IM_STUDY_ID,
           'im_acquired_study_id', a.IM_ACQUIRED_STUDY_ID, 'encntr_id', p.ENCNTR_ID, 'order_id', p.ORDER_ID,
           'mill_accession', p.ACCESSION, 'proc_status_cd', p.PROC_STATUS_CD, 'trust', p.Trust,
           'action_dt_tm', p.ACTION_DT_TM) AS {ENRICHMENT_TYPE}) AS enrichment
  FROM 4_prod.raw.mill_cv_proc p
  JOIN 4_prod.raw.mill_im_study s
    ON s.ORIG_ENTITY_ID = p.CV_PROC_ID AND s.ORIG_ENTITY_NAME = 'CV_PROC'
  JOIN 4_prod.raw.mill_im_acquired_study a
    ON a.MATCHED_STUDY_ID = s.IM_STUDY_ID
  WHERE p.CV_PROC_ID <> 0
    AND coalesce(a.IM_DEVICE_ID, 0) <> 0
    AND coalesce(trim(s.STUDY_UID), '') <> ''
),
r AS (
  SELECT *, row_number() OVER (
           PARTITION BY study_instance_uid, device_id
           ORDER BY CASE WHEN source_updt_dt_tm IS NULL THEN 1 ELSE 0 END,
                    source_updt_dt_tm DESC NULLS LAST,
                    im_acquired_study_id DESC NULLS LAST,
                    cv_proc_id DESC NULLS LAST) AS rn
  FROM j
)
SELECT study_instance_uid, device_id, action_dt_tm, source_updt_dt_tm, enrichment,
       sha2(to_json(enrichment), 256) AS enrichment_hash,
       concat(study_instance_uid, '__', CAST(device_id AS STRING)) AS file_stem
FROM r
WHERE rn = 1
"""


def setup():
    spark.sql(MANIFEST_DDL)
    spark.sql(CANDIDATES_DDL)
    dup = spark.sql(f"""SELECT count(*) - count(DISTINCT study_instance_uid, device_id) AS d
                        FROM {CANDIDATES}""").collect()[0]["d"]
    assert dup == 0, f"candidates view is not unique on (study_instance_uid, device_id): {dup}"

In [0]:
# IG tags: every column tagged (identifiers stay in bronze; de-identification is serve-time).
TAGS_4_2 = {
    "STUDY_INSTANCE_UID", "DICOM_STUDY_INSTANCE_UID", "SOP_INSTANCE_UID", "SERIES_INSTANCE_UID",
    "PATIENT_NAME", "DICOM_PATIENT_ID", "PATIENT_BIRTH_DATE_RAW", "PATIENT_BIRTH_DT_TM",
    "PATIENT_BIRTH_DT_TM_CLEAN", "PATIENT_BIRTH_DT_TM_FUTURE_IND", "PATIENT_BIRTH_DT_TM_SENTINEL_IND",
    "ACCESSION_NBR", "MILL_ACCESSION", "REFERRING_PHYSICIAN_NAME", "OPERATORS_NAME",
    "INSTITUTION_NAME", "INSTITUTIONAL_DEPARTMENT_NAME", "FILE_PATH", "FILE_SHA256",
}
TAGS_3_2 = {"UNFORMATTED_TEXT_VALUE", "PARSE_ERROR"}
TAGS_1_1 = {"TRUST", "SAMPLES", "PATIENT_AGE_RAW", "PATIENT_SIZE_M", "PATIENT_WEIGHT_KG"}
TAGS_0_1 = {"PERSON_ID"}


def ig_tag(column):
    if column in TAGS_4_2:
        return "4", "2"
    if column in TAGS_3_2:
        return "3", "2"
    if column in TAGS_1_1:
        return "1", "1"
    if column in TAGS_0_1:
        return "0", "1"
    return "0", "0"


def _current_ig_tags(table):
    catalog, schema, name = table.split(".")
    tags = {}
    for r in spark.sql(f"""
            SELECT column_name, tag_name, tag_value
            FROM {catalog}.information_schema.column_tags
            WHERE schema_name = '{schema}' AND table_name = '{name}'
              AND tag_name IN ('ig_risk', 'ig_severity')""").collect():
        tags[(r["column_name"].upper(), r["tag_name"])] = r["tag_value"]
    return tags


def ensure_ig_tags(table):
    """Reconcile ig_risk AND ig_severity on every column; runs before every publication,
    so a half-tagged table left by a failed earlier run is repaired, never written to."""
    columns = spark.table(table).columns
    current = _current_ig_tags(table)
    for column in columns:
        risk, severity = ig_tag(column.upper())
        if (current.get((column.upper(), "ig_risk")), current.get((column.upper(), "ig_severity"))) != (risk, severity):
            spark.sql(f"ALTER TABLE {table} ALTER COLUMN {column} "
                      f"SET TAGS ('ig_risk' = '{risk}', 'ig_severity' = '{severity}')")
    current = _current_ig_tags(table)
    wrong = [c for c in columns
             if (current.get((c.upper(), "ig_risk")), current.get((c.upper(), "ig_severity"))) != ig_tag(c.upper())]
    assert not wrong, f"{table}: ig_risk/ig_severity missing or wrong on {wrong}"

In [0]:
# Reconcile: a claimed row is downloaded only when its file is in batches/<its own batch_id>/.
# A previous claim's file lives in a different folder, so it can never satisfy a refetch.
BATCH_ID_SAFE = re.compile(r"^[0-9A-Za-z-]+$")


def reconcile():
    batches = [r["batch_id"] for r in spark.sql(
        f"SELECT DISTINCT batch_id FROM {MANIFEST} WHERE status = 'claimed' AND batch_id IS NOT NULL").collect()]
    rows = []
    for batch_id in batches:
        path = os.path.join(LANDING_BASE, "batches", batch_id)
        if not BATCH_ID_SAFE.match(batch_id) or not os.path.isdir(path):
            continue
        with os.scandir(path) as it:
            for entry in it:
                parsed = ecg_dicom.parse_landing_name(entry.name)
                if parsed is None or not entry.is_file():
                    continue
                mtime = datetime.fromtimestamp(entry.stat().st_mtime, tz=timezone.utc)
                rows.append((batch_id, parsed[0], entry.path, parsed[1], parsed[2], mtime))
    landed = 0
    if rows:
        spark.createDataFrame(rows, "batch_id string, file_stem string, file_path string, "
                                    "instance_number int, instance_count int, file_mtime timestamp"
                              ).createOrReplaceTempView("ecg_landed_files")
        landed = spark.sql(f"""
            MERGE INTO {MANIFEST} t
            USING (
              SELECT m.study_instance_uid, m.device_id, m.batch_id, f.file_path, f.instance_number,
                     f.instance_count, f.file_mtime,
                     row_number() OVER (PARTITION BY m.study_instance_uid, m.device_id
                                        ORDER BY f.instance_number DESC NULLS LAST,
                                                 f.file_mtime DESC NULLS LAST, f.file_path DESC) AS rn
              FROM {MANIFEST} m
              JOIN ecg_landed_files f
                ON f.batch_id = m.batch_id AND f.file_stem = m.file_stem
              WHERE m.status = 'claimed'
            ) s
            ON t.study_instance_uid = s.study_instance_uid AND t.device_id = s.device_id
               AND s.rn = 1 AND t.status = 'claimed' AND t.batch_id = s.batch_id
            WHEN MATCHED THEN UPDATE SET
              t.status = 'downloaded', t.file_path = s.file_path,
              t.instance_number = s.instance_number, t.instance_count = s.instance_count,
              t.downloaded_at = s.file_mtime, t.last_error = NULL, t.updated_at = current_timestamp()
        """).collect()[0]["num_updated_rows"]
    expired = spark.sql(f"""
        UPDATE {MANIFEST}
        SET status = CASE WHEN attempts >= {MAX_ATTEMPTS} THEN 'failed_permanent' ELSE 'pending' END,
            last_error = concat('no landed file within {CLAIM_TIMEOUT_HOURS}h of claim, batch ', batch_id),
            updated_at = current_timestamp()
        WHERE status = 'claimed'
          AND claimed_at < current_timestamp() - INTERVAL {CLAIM_TIMEOUT_HOURS} HOURS
    """).collect()[0]["num_affected_rows"]
    return {"batches_listed": len(batches), "files_seen": len(rows),
            "claims_landed": int(landed), "claims_expired": int(expired)}

In [0]:
# Parse: snapshot the downloaded rows (CLAIMS), parse their files into flat union records (STAGE).
STAGE_SCHEMA = "STUDY_INSTANCE_UID string, DEVICE_ID bigint, " + ecg_dicom.RECORD_SCHEMA
STAGE_COLUMNS = ["STUDY_INSTANCE_UID", "DEVICE_ID"] + ecg_dicom.RECORD_COLUMNS


def _parse_partition(batches):
    out = []
    for pdf in batches:
        for row in pdf.itertuples(index=False):
            try:
                with open(row.file_path, "rb") as fh:
                    records = ecg_dicom.parse_ecg(fh.read(), row.file_path)
            except OSError as exc:
                # The landed file is gone: never publish over bronze; mark_parsed re-queues a fetch.
                rec = dict.fromkeys(ecg_dicom.RECORD_COLUMNS)
                rec.update(RECORD_TYPE="study", FILE_PATH=row.file_path, PARSE_STATUS="file_missing",
                           PARSE_ERROR=f"{type(exc).__name__}: {exc}"[:1000])
                records = [rec]
            for rec in records:
                out.append({"STUDY_INSTANCE_UID": row.study_instance_uid,
                            "DEVICE_ID": row.device_id, **rec})
            if len(out) >= FLUSH_RECORDS:
                yield pd.DataFrame(out, columns=STAGE_COLUMNS, dtype=object)
                out = []
    if out:
        yield pd.DataFrame(out, columns=STAGE_COLUMNS, dtype=object)


def parse_to_stage():
    # Snapshot, so enrichment and claim identity cannot shift under this run (the seed may update
    # the manifest concurrently); mark_parsed completes only rows still on this snapshot's claim.
    spark.sql(f"""
        CREATE OR REPLACE TABLE {CLAIMS} AS
        SELECT study_instance_uid, device_id, batch_id, file_path, instance_number, instance_count,
               downloaded_at, refetch_count, claim_source_updt_dt_tm, enrichment, enrichment_hash
        FROM {MANIFEST}
        WHERE status = 'downloaded' AND file_path IS NOT NULL
        ORDER BY downloaded_at, study_instance_uid, device_id
        LIMIT {MAX_FILES}""")
    todo = spark.table(CLAIMS).select("study_instance_uid", "device_id", "file_path")
    n = todo.count()
    if n == 0:
        return 0
    (todo.repartition(max(1, n // 200))
         .mapInPandas(_parse_partition, STAGE_SCHEMA)
         .withColumn("PARSER_VERSION", F.lit(ecg_dicom.PARSER_VERSION))
         .write.mode("overwrite").option("overwriteSchema", "true").saveAsTable(STAGE))
    staged = spark.sql(f"""SELECT count(DISTINCT STUDY_INSTANCE_UID, DEVICE_ID) AS k,
                                  count_if(RECORD_TYPE = 'study') AS s FROM {STAGE}""").collect()[0]
    assert staged["k"] == n and staged["s"] == n, (
        f"stage lost ECGs: todo={n} keys={staged['k']} study_records={staged['s']}")
    return n


def choose_publications():
    """PUBLISH = the ECG generations this run writes. A missing file never publishes; a failed
    parse never overwrites an existing good generation (bronze keeps it, the manifest records why)."""
    keep_good = ""
    if spark.catalog.tableExists(T_STUDY):
        keep_good = f"""AND NOT (s.PARSE_STATUS = 'failed' AND EXISTS (
                          SELECT 1 FROM {T_STUDY} b
                          WHERE b.STUDY_INSTANCE_UID = s.STUDY_INSTANCE_UID AND b.DEVICE_ID = s.DEVICE_ID
                            AND b.PARSE_STATUS IN ('ok', 'no_waveform')))"""
    spark.sql(f"""
        CREATE OR REPLACE TABLE {PUBLISH} AS
        SELECT s.STUDY_INSTANCE_UID, s.DEVICE_ID, s.FILE_SHA256, s.PARSE_STATUS
        FROM {STAGE} s
        WHERE s.RECORD_TYPE = 'study' AND s.PARSE_STATUS <> 'file_missing' {keep_good}""")
    return spark.table(PUBLISH).count()

In [0]:
# Build the three bronze frames from STAGE + the CLAIMS snapshot, restricted to PUBLISH.
def dq_columns(df, date_cols):
    """Master plan 2.2 date-quality block: C_FUTURE_IND, C_SENTINEL_IND, C_CLEAN per timestamp C."""
    out = df
    for c in date_cols:
        fut = F.col(c) > F.current_timestamp()
        sen = F.col(c) < F.lit(SENTINEL_FLOOR).cast("timestamp")
        out = (out
               .withColumn(f"{c}_FUTURE_IND", F.when(F.col(c).isNull(), F.lit(None)).otherwise(fut))
               .withColumn(f"{c}_SENTINEL_IND", F.when(F.col(c).isNull(), F.lit(None)).otherwise(sen))
               .withColumn(f"{c}_CLEAN", F.when(fut | sen, F.lit(None).cast("timestamp")).otherwise(F.col(c))))
    return out


def _row_hash(df, exclude=("PIPELINE_UPDT_DT_TM", "SAMPLES")):
    # SAMPLES is covered by FILE_SHA256 (same bytes -> same samples) plus PARSER_VERSION.
    cols = [c for c in df.columns if c not in exclude]
    return df.withColumn("ROW_HASH", F.sha2(F.concat_ws("||", *[F.coalesce(F.col(c).cast("string"), F.lit("")) for c in cols]), 256))


def build_study():
    header = ", ".join(f"s.{name}" for name, _ in ecg_dicom.STUDY_FIELDS)
    df = spark.sql(f"""
        WITH c AS (
          SELECT c.* FROM {CLAIMS} c
          JOIN {PUBLISH} p ON p.STUDY_INSTANCE_UID = c.study_instance_uid AND p.DEVICE_ID = c.device_id
        ),
        s AS (
          SELECT * FROM {STAGE} WHERE RECORD_TYPE = 'study'
        ),
        cv AS (
          SELECT CAST(CODE_VALUE AS BIGINT) AS CODE_VALUE, max(DESCRIPTION) AS DESCRIPTION
          FROM 3_lookup.mill.mill_code_value GROUP BY 1
        ),
        mrn AS (
          SELECT DISTINCT CAST(a.PERSON_ID AS BIGINT) AS PERSON_ID, upper(trim(a.ALIAS)) AS ALIAS
          FROM 4_prod.raw.mill_person_alias a
          JOIN (SELECT DISTINCT enrichment.person_id AS person_id FROM c) p
            ON CAST(a.PERSON_ID AS BIGINT) = p.person_id
          WHERE CAST(a.PERSON_ALIAS_TYPE_CD AS BIGINT) = {MRN_ALIAS_TYPE_CD} AND CAST(a.ACTIVE_IND AS INT) = 1
        )
        SELECT c.study_instance_uid AS STUDY_INSTANCE_UID,
               c.device_id AS DEVICE_ID,
               c.enrichment.im_study_id AS IM_STUDY_ID,
               c.enrichment.im_acquired_study_id AS IM_ACQUIRED_STUDY_ID,
               c.enrichment.cv_proc_id AS CV_PROC_ID,
               c.enrichment.person_id AS PERSON_ID,
               c.enrichment.encntr_id AS ENCNTR_ID,
               c.enrichment.order_id AS ORDER_ID,
               c.enrichment.mill_accession AS MILL_ACCESSION,
               c.enrichment.proc_status_cd AS PROC_STATUS_CD,
               cv.DESCRIPTION AS PROC_STATUS_DESC,
               c.enrichment.action_dt_tm AS ACTION_DT_TM,
               -- DICOM times are device-local (UK) with no offset; stored as UTC like the Millennium *_DT_TM columns
               to_utc_timestamp(
                 coalesce(try_to_timestamp(substr(s.ACQUISITION_DATETIME_RAW, 1, 14), 'yyyyMMddHHmmss'),
                          try_to_timestamp(concat(s.STUDY_DATE_RAW, substr(rpad(coalesce(s.STUDY_TIME_RAW, ''), 6, '0'), 1, 6)),
                                           'yyyyMMddHHmmss')),
                 'Europe/London') AS ACQUISITION_DT_TM,
               c.enrichment.trust AS TRUST,
               CASE WHEN s.DICOM_PATIENT_ID IS NULL OR c.enrichment.person_id IS NULL THEN NULL
                    WHEN mrn.ALIAS IS NOT NULL THEN 1 ELSE 0 END AS DICOM_PATIENT_ID_MRN_MATCH_IND,
               {header},
               try_to_timestamp(s.PATIENT_BIRTH_DATE_RAW, 'yyyyMMdd') AS PATIENT_BIRTH_DT_TM,
               c.instance_number AS CAMM_INSTANCE_NUMBER,
               c.instance_count AS CAMM_INSTANCE_COUNT,
               c.batch_id AS CAMM_CLAIM_ID,
               c.claim_source_updt_dt_tm AS SOURCE_UPDT_DT_TM,
               c.enrichment_hash AS ENRICHMENT_HASH,
               c.file_path AS FILE_PATH,
               c.downloaded_at AS DOWNLOADED_DT_TM,
               c.refetch_count AS REFETCH_COUNT,
               s.PARSER_VERSION AS PARSER_VERSION
        FROM c
        JOIN s ON s.STUDY_INSTANCE_UID = c.study_instance_uid AND s.DEVICE_ID = c.device_id
        LEFT JOIN cv ON cv.CODE_VALUE = c.enrichment.proc_status_cd
        LEFT JOIN mrn ON mrn.PERSON_ID = c.enrichment.person_id AND mrn.ALIAS = upper(trim(s.DICOM_PATIENT_ID))
    """)
    df = dq_columns(df, ["ACTION_DT_TM", "ACQUISITION_DT_TM", "PATIENT_BIRTH_DT_TM"])
    df = df.withColumn("PIPELINE_UPDT_DT_TM", F.current_timestamp())
    return _row_hash(df)


def build_detail(record_type, fields):
    cols = ", ".join(f"d.{name}" for name, _ in fields)
    df = spark.sql(f"""
        SELECT d.STUDY_INSTANCE_UID, d.DEVICE_ID, c.enrichment.person_id AS PERSON_ID, {cols},
               p.FILE_SHA256, d.PARSER_VERSION
        FROM {STAGE} d
        JOIN {PUBLISH} p
          ON p.STUDY_INSTANCE_UID = d.STUDY_INSTANCE_UID AND p.DEVICE_ID = d.DEVICE_ID
        JOIN {CLAIMS} c
          ON c.study_instance_uid = d.STUDY_INSTANCE_UID AND c.device_id = d.DEVICE_ID
        WHERE d.RECORD_TYPE = '{record_type}'
    """).withColumn("PIPELINE_UPDT_DT_TM", F.current_timestamp())
    return _row_hash(df)

In [0]:
# Publish. Each table changes in ONE atomic MERGE per run; detail tables are generation-keyed
# by FILE_SHA256, so readers joining detail -> study on (STUDY_INSTANCE_UID, DEVICE_ID, FILE_SHA256)
# always see a complete generation:
#   1. upsert the new generation into waveform + annotation (old generation still present)
#   2. flip map_ecg_study to the new generation (one MERGE: the commit point)
#   3. retire superseded generations from waveform + annotation
# A crash anywhere leaves the manifest at 'downloaded'; the rerun repeats 1-3 idempotently
# (ROW_HASH-equal rows are not rewritten).
ECG_KEY = ["STUDY_INSTANCE_UID", "DEVICE_ID"]
TABLE_SPECS = {
    T_STUDY: {
        "row_key": ECG_KEY,
        "scope": ECG_KEY,
        "cluster": ["PERSON_ID", "ACQUISITION_DT_TM"],
        "comment": "Bronze ECG study: one row per CAMM 12-lead ECG DICOM (study UID x device) with DICOM header, "
                   "Millennium CV_PROC enrichment and parse provenance. Grain STUDY_INSTANCE_UID, DEVICE_ID. "
                   "FILE_SHA256 names the published generation: join map_ecg_waveform / map_ecg_annotation on "
                   "(STUDY_INSTANCE_UID, DEVICE_ID, FILE_SHA256)."},
    T_WAVEFORM: {
        "row_key": ECG_KEY + ["FILE_SHA256", "WAVEFORM_GROUP_INDEX", "CHANNEL_INDEX"],
        "scope": ECG_KEY + ["FILE_SHA256"],
        "cluster": ["PERSON_ID", "STUDY_INSTANCE_UID"],
        "comment": "Bronze ECG waveform: one row per ECG x waveform group x channel. SAMPLES are stored integers; "
                   "physical value = " + ecg_dicom.SCALING_FORMULA + ". Join to map_ecg_study on "
                   "(STUDY_INSTANCE_UID, DEVICE_ID, FILE_SHA256); rows of a superseded generation may exist "
                   "briefly during a publish."},
    T_ANNOTATION: {
        "row_key": ECG_KEY + ["FILE_SHA256", "ANNOTATION_INDEX"],
        "scope": ECG_KEY + ["FILE_SHA256"],
        "cluster": ["PERSON_ID", "STUDY_INSTANCE_UID"],
        "comment": "Bronze ECG annotation: device measurements (coded + numeric) and interpretation text, one row per "
                   "WaveformAnnotationSequence item. Join to map_ecg_study on (STUDY_INSTANCE_UID, DEVICE_ID, FILE_SHA256)."},
}


def _on(cols, left, right):
    return " AND ".join(f"{left}.{c} <=> {right}.{c}" for c in cols)


def prepare_table(table, df):
    """Create on first run (empty, so nothing is ever published untagged), then reconcile tags."""
    spec = TABLE_SPECS[table]
    if not spark.catalog.tableExists(table):
        # Liquid clustering needs stats on its keys; Delta collects stats on the first 32 columns only
        outside = [c for c in spec["cluster"] if c not in df.columns[:32]]
        assert not outside, f"{table}: clustering column(s) {outside} are outside the first 32 (stats) columns"
        df.limit(0).createOrReplaceTempView("ecg_empty")
        spark.sql(f"""CREATE TABLE {table} CLUSTER BY ({', '.join(spec['cluster'])})
                      COMMENT '{spec['comment']}'
                      TBLPROPERTIES ('delta.enableChangeDataFeed' = 'true')
                      AS SELECT * FROM ecg_empty""")
    ensure_ig_tags(table)
    existing = spark.table(table).columns
    assert set(df.columns) == set(existing), (
        f"{table}: schema drift {sorted(set(df.columns) ^ set(existing))}; migrate the table first")
    return existing


def upsert_generation(table, df):
    spec = TABLE_SPECS[table]
    columns = prepare_table(table, df)
    key = spec["row_key"]
    chk = df.agg(F.count(F.lit(1)).alias("n"), F.countDistinct(*key).alias("k")).collect()[0]
    assert chk["n"] == chk["k"], f"{table}: {chk['n']} rows for {chk['k']} row keys"
    view = "ecg_new_" + table.split(".")[-1]
    df.select(*columns).createOrReplaceTempView(view)
    col_list = ", ".join(columns)
    spark.sql(f"""
        MERGE INTO {table} t
        USING (
          SELECT {col_list}, false AS _ECG_DELETE FROM {view}
          UNION ALL
          SELECT {', '.join('o.' + c for c in columns)}, true AS _ECG_DELETE
          FROM {table} o
          LEFT SEMI JOIN (SELECT DISTINCT {', '.join(spec['scope'])} FROM {view}) g
            ON {_on(spec['scope'], 'o', 'g')}
          LEFT ANTI JOIN {view} n ON {_on(key, 'o', 'n')}
        ) s
        ON {_on(key, 't', 's')}
        WHEN MATCHED AND s._ECG_DELETE THEN DELETE
        WHEN MATCHED AND NOT (t.ROW_HASH <=> s.ROW_HASH) THEN UPDATE SET {', '.join(f't.{c} = s.{c}' for c in columns)}
        WHEN NOT MATCHED AND NOT s._ECG_DELETE THEN INSERT ({col_list}) VALUES ({', '.join('s.' + c for c in columns)})""")


def retire_superseded(table):
    spark.sql(f"""
        MERGE INTO {table} t
        USING {PUBLISH} p
        ON t.STUDY_INSTANCE_UID = p.STUDY_INSTANCE_UID AND t.DEVICE_ID = p.DEVICE_ID
           AND NOT (t.FILE_SHA256 <=> p.FILE_SHA256)
        WHEN MATCHED THEN DELETE""")


def mark_parsed():
    # Completes only rows still on the snapshot's claim + file; anything the seed re-queued or
    # ADF re-claimed meanwhile is left alone. applied_enrichment_hash moves only when published.
    spark.sql(f"""
        MERGE INTO {MANIFEST} t
        USING (
          SELECT c.study_instance_uid, c.device_id, c.batch_id, c.file_path, c.enrichment_hash,
                 s.PARSE_STATUS, s.PARSE_ERROR, s.PARSER_VERSION,
                 p.STUDY_INSTANCE_UID IS NOT NULL AS published
          FROM {CLAIMS} c
          JOIN {STAGE} s
            ON s.STUDY_INSTANCE_UID = c.study_instance_uid AND s.DEVICE_ID = c.device_id AND s.RECORD_TYPE = 'study'
          LEFT JOIN {PUBLISH} p
            ON p.STUDY_INSTANCE_UID = c.study_instance_uid AND p.DEVICE_ID = c.device_id
        ) s
        ON t.study_instance_uid = s.study_instance_uid AND t.device_id = s.device_id
           AND t.status = 'downloaded' AND t.batch_id <=> s.batch_id AND t.file_path <=> s.file_path
        WHEN MATCHED AND s.PARSE_STATUS = 'file_missing' THEN UPDATE SET
          t.status = 'pending', t.file_path = NULL, t.last_error = s.PARSE_ERROR, t.updated_at = current_timestamp()
        WHEN MATCHED THEN UPDATE SET
          t.status = CASE WHEN s.PARSE_STATUS = 'failed' THEN 'parse_failed' ELSE 'parsed' END,
          t.parsed_at = current_timestamp(), t.parser_version = s.PARSER_VERSION,
          t.applied_enrichment_hash = CASE WHEN s.published THEN s.enrichment_hash ELSE t.applied_enrichment_hash END,
          t.last_error = CASE WHEN s.published THEN s.PARSE_ERROR
                              ELSE concat('kept previous bronze generation; ', s.PARSE_ERROR) END,
          t.updated_at = current_timestamp()""")

In [0]:
def ingest():
    result = {"action": "ingest", "adf_run_id": ADF_RUN_ID, "target_schema": TARGET_SCHEMA}
    result.update(reconcile())
    result["parsed_files"] = parse_to_stage()
    if result["parsed_files"]:
        result["published"] = choose_publications()
        waveform = build_detail("waveform", ecg_dicom.WAVEFORM_FIELDS)
        annotation = build_detail("annotation", ecg_dicom.ANNOTATION_FIELDS)
        study = build_study()
        upsert_generation(T_WAVEFORM, waveform)
        upsert_generation(T_ANNOTATION, annotation)
        upsert_generation(T_STUDY, study)
        retire_superseded(T_WAVEFORM)
        retire_superseded(T_ANNOTATION)
        mark_parsed()
        for scratch in (STAGE, CLAIMS, PUBLISH):
            spark.sql(f"TRUNCATE TABLE {scratch}")
    result["manifest"] = {r["status"]: r["n"] for r in spark.sql(
        f"SELECT status, count(*) AS n FROM {MANIFEST} GROUP BY status").collect()}
    return result


if ACTION == "setup":
    setup()
    RESULT = {"action": "setup", "manifest": MANIFEST, "candidates": CANDIDATES}
elif ACTION == "reparse_failed":
    n = spark.sql(f"""UPDATE {MANIFEST} SET status = 'downloaded', updated_at = current_timestamp()
                      WHERE status = 'parse_failed' AND file_path IS NOT NULL""").collect()[0]["num_affected_rows"]
    RESULT = {"action": "reparse_failed", "reset": int(n)}
else:
    RESULT = ingest()
print(json.dumps(RESULT, default=str))
dbutils.notebook.exit(json.dumps(RESULT, default=str))